# 第 3 讲：语言模型架构与超参数

这是 CS336 Lecture 03 的可执行中文学习版。

- 官方来源：`external/cs336-lectures/lecture_03.pdf`
- 官方形式：PDF lecture
- 英文标题：LM Architecture And Hyperparameters

这份 notebook 是学习版，不是逐字翻译。它按官方讲义的结构和节奏整理主线，并在适合的位置加入小实验。


## 0. 本讲你要抓住什么

这一讲从现代 Transformer 的常见组件出发，讨论大模型架构里哪些选择已经变成默认项，哪些仍然依赖实验经验。

学习方式：

1. 先读每节的中文解释。
2. 运行对应代码 cell。
3. 改一个参数，观察输出如何变化。
4. 最后用检查点问题自测。


## 1. 本讲路线图

复习现代 Transformer，再看 RoPE、pre-norm、SwiGLU、bias、normalization、宽深比例和训练超参数。


In [1]:
lecture = 3
title = 'LM Architecture And Hyperparameters'
source = 'external/cs336-lectures/lecture_03.pdf'
print(f"Lecture {lecture}: {title}")
print("source:", source)


Lecture 3: LM Architecture And Hyperparameters
source: external/cs336-lectures/lecture_03.pdf


## 2. 原始 Transformer 到现代变体

现代 decoder-only LM 通常使用 causal self-attention、pre-norm、RoPE、SwiGLU 和无 bias 的线性层。


## 3. Normalization

LayerNorm 和 RMSNorm 都用于稳定 activation 尺度；RMSNorm 更简单，少了均值中心化。


## 4. 位置编码

RoPE 把位置信息注入 Q/K 的旋转结构中，让相对位置信息影响 attention 分数。


## 5. FFN 与 SwiGLU

现代 LM 常用 gated FFN，例如 SwiGLU，用门控控制信息流。


## 6. 超参数不是装饰

depth、width、heads、learning rate、batch size、初始化都会影响稳定性、效率和最终 loss。


## 动手实验 1：RMSNorm 与 LayerNorm

先直接运行，再改一个数字或字符串。


In [2]:
import numpy as np

x = np.array([1.0, 2.0, 10.0])
layer_norm = (x - x.mean()) / np.sqrt(((x - x.mean()) ** 2).mean() + 1e-5)
rms_norm = x / np.sqrt((x ** 2).mean() + 1e-5)
print("x:        ", x)
print("LayerNorm:", layer_norm)
print("RMSNorm:  ", rms_norm)


x:         [ 1.  2. 10.]
LayerNorm: [-0.82760563 -0.57932394  1.40692958]
RMSNorm:   [0.16903083 0.33806165 1.69030827]


## 动手实验 2：SwiGLU 的门控直觉

先直接运行，再改一个数字或字符串。


In [3]:
import math

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def silu(x):
    return x * sigmoid(x)

xs = [-3, -1, 0, 1, 3]
for x in xs:
    gate = silu(x)
    value = x
    print(x, "gate=", round(gate, 3), "gated value=", round(gate * value, 3))


-3 gate= -0.142 gated value= 0.427
-1 gate= -0.269 gated value= 0.269
0 gate= 0.0 gated value= 0.0
1 gate= 0.731 gated value= 0.731
3 gate= 2.858 gated value= 8.573


## 动手实验 3：RoPE 的二维旋转玩具版

先直接运行，再改一个数字或字符串。


In [4]:
import math
import numpy as np

def rotate2d(v, position, theta=0.2):
    angle = position * theta
    R = np.array([[math.cos(angle), -math.sin(angle)], [math.sin(angle), math.cos(angle)]])
    return R @ np.array(v)

q = [1.0, 0.0]
for pos in range(5):
    print(pos, rotate2d(q, pos))


0 [1. 0.]
1 [0.98006658 0.19866933]
2 [0.92106099 0.38941834]
3 [0.82533561 0.56464247]
4 [0.69670671 0.71735609]


## 今日检查点

1. 能画出 decoder-only Transformer block 的主要路径。
2. 能解释 pre-norm 为什么和训练稳定性有关。
3. 能说出 RoPE 在什么位置作用。
4. 能区分 architecture choice 和 hyperparameter choice。

如果这些能讲清楚，这一讲的第一轮学习就完成了。
